In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier
)


models = {
            "Random Forest": RandomForestClassifier(verbose=1,n_jobs=-1),
            "Decision Tree": DecisionTreeClassifier(),
            "Gradient Boosting": GradientBoostingClassifier(verbose=1),
            "Logistic Regression": LogisticRegression(verbose=1,n_jobs=-1),
            "AdaBoost": AdaBoostClassifier(),
        }

params={
    "Decision Tree": {
        'criterion':['gini', 'entropy', 'log_loss'],
        # 'splitter':['best','random'],
        # 'max_features':['sqrt','log2'],
    },
    "Random Forest":{
        # 'criterion':['gini', 'entropy', 'log_loss'],
        
        # 'max_features':['sqrt','log2',None],
        'n_estimators': [8,16,32,128,256]
    },
    "Gradient Boosting":{
        # 'loss':['log_loss', 'exponential'],
        'learning_rate':[.1,.01,.05,.001],
        'subsample':[0.6,0.7,0.75,0.85,0.9],
        # 'criterion':['squared_error', 'friedman_mse'],
        # 'max_features':['auto','sqrt','log2'],
        'n_estimators': [8,16,32,64,128,256]
    },
    "Logistic Regression":{},
    "AdaBoost":{
        'learning_rate':[.1,.01,.001],
        'n_estimators': [8,16,32,64,128,256]
    }
    
}

In [2]:
model = list(models.values())[0]
print(model)

RandomForestClassifier(n_jobs=-1, verbose=1)


In [3]:
param = list(params.keys())
params[param[0]]

{'criterion': ['gini', 'entropy', 'log_loss']}

In [4]:
for key in models.keys():
    model = models[key]
    param = params[key]
print(model)
print(param)

AdaBoostClassifier()
{'learning_rate': [0.1, 0.01, 0.001], 'n_estimators': [8, 16, 32, 64, 128, 256]}


## Random search CV
Tune each model with `RandomizedSearchCV`: it samples `n_iter` random combinations from `params` instead of trying the full grid.

In [5]:
import numpy as np
from pathlib import Path

# latest transformed artifacts; last column = target
latest = sorted(Path("../Artifacts").glob("*/data_transformation/transformed"))[-1]
train_arr = np.load(latest / "train.npy")
test_arr = np.load(latest / "test.npy")

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]
print(latest, X_train.shape, X_test.shape)

../Artifacts/11_18_2026_16_39_53/data_transformation/transformed (8844, 30) (2211, 30)


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score

# quiet the verbose estimators for the search
for m in models.values():
    if hasattr(m, "set_params") and "verbose" in m.get_params():
        m.set_params(verbose=0)

results = {}
for name, model in models.items():
    grid = params[name]
    if not grid:  # nothing to tune (e.g. Logistic Regression)
        model.fit(X_train, y_train)
        results[name] = {"cv_f1": None, "test_f1": f1_score(y_test, model.predict(X_test)), "best_params": {}}
        continue

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=grid,
        n_iter=10,          # random combos sampled per model
        cv=3,
        scoring="f1",
        n_jobs=-1,
        random_state=42,
    )
    search.fit(X_train, y_train)
    results[name] = {
        "cv_f1": search.best_score_,
        "test_f1": f1_score(y_test, search.best_estimator_.predict(X_test)),
        "best_params": search.best_params_,
    }

for name, r in results.items():
    print(name, r)

/home/surya/ml-proj/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:327: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


/home/surya/ml-proj/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:327: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


/home/surya/ml-proj/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Random Forest {'cv_f1': np.float64(0.9704066215374393), 'test_f1': 0.9768009768009768, 'best_params': {'n_estimators': 256}}
Decision Tree {'cv_f1': np.float64(0.961145045438259), 'test_f1': 0.9644171779141104, 'best_params': {'criterion': 'entropy'}}
Gradient Boosting {'cv_f1': np.float64(0.9629129831847139), 'test_f1': 0.967059780398536, 'best_params': {'subsample': 0.7, 'n_estimators': 256, 'learning_rate': 0.1}}
Logistic Regression {'cv_f1': None, 'test_f1': 0.9399837793998378, 'best_params': {}}
AdaBoost {'cv_f1': np.float64(0.9331025621792914), 'test_f1': 0.9395758303321329, 'best_params': {'n_estimators': 256, 'learning_rate': 0.1}}


In [7]:
import pandas as pd
pd.DataFrame(results).T.sort_values("test_f1", ascending=False)

,cv_f1,test_f1,best_params
Random Forest,0.970407,0.976801,{'n_estimators': 256}
Gradient Boosting,0.962913,0.96706,"{'subsample': 0.7, 'n_estimators': 256, 'learn..."
Decision Tree,0.961145,0.964417,{'criterion': 'entropy'}
Logistic Regression,None,0.939984,{}
AdaBoost,0.933103,0.939576,"{'n_estimators': 256, 'learning_rate': 0.1}"
